In [ ]:
import pandas as pd

# Step 1: Read CSV
df = pd.read_csv("date_earnings.csv")

# Step 2: Convert 'Date' to datetime objects
df['date'] = pd.to_datetime(df['date'], dayfirst=True)
df['date'] = df['date'] + pd.DateOffset(months=1)

# Step 3: Clean numeric columns (all states + Australia)
for col in df.columns[1:]:  # skip 'Date'
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', ''), errors='coerce')

# Step 4: Create a quarterly date range spanning the data
quarterly_dates = pd.date_range(start=df['date'].min(),
                                end=pd.Timestamp('2026-01-01'),
                                freq='QS') - pd.DateOffset(months=1)

# Step 5: Set 'Date' as index
df.set_index('date', inplace=True)

# Step 6: Reindex to quarterly dates and interpolate each column
df = df.reindex(quarterly_dates)
df = df.interpolate(method='linear')

# Step 7: Reset index and rename
df.reset_index(inplace=True)
df.rename(columns={'index': 'date'}, inplace=True)

# Step 8: Round for easier reading
df[df.columns[1:]] = df[df.columns[1:]].round()

df = df.drop(columns=['Unnamed: 10'])
# Step 9: View results
display(df)

In [ ]:
df_melt = df.melt(
    id_vars='date',
    var_name='state',
    value_name='earnings'
)

df_melt['state'] = (
    df_melt['state']
    .str.replace('_earnings', '', regex=False)
    .replace({'aus': 'all'})
)

In [ ]:
# --- Load Excel (second tab: Data1) ---
df = pd.read_excel('cpi_index.xlsx', sheet_name='Data1',
    skiprows=9, usecols="A:J")

# --- Rename columns (match order in your file) ---
df.columns = [
    'date', 'sydney', 'melbourne', 'brisbane', 'adelaide',
    'perth', 'hobart', 'darwin', 'canberra', 'all'
]

# --- Convert date ---
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# --- Drop rows with invalid dates (from header junk etc.) ---
df = df.dropna(subset=['date'])

# --- Filter date range ---
df = df[df['date'] >= '2010-06-01']

# --- OPTIONAL: drop columns you don’t care about ---
# df = df.drop(columns=['darwin', 'hobart'])

# Melt to long format
df_cpi = df.melt(
    id_vars='date',
    var_name='state',
    value_name='cpi'
)

In [ ]:
state_to_city = {
    'nsw': 'sydney',
    'vic': 'melbourne',
    'qld': 'brisbane',
    'sa': 'adelaide',
    'wa': 'perth',
    'tas': 'hobart',
    'nt': 'darwin',
    'act': 'canberra',
    'all': 'all'
}

df_melt['state'] = df_melt['state'].replace(state_to_city)

df_merged = df_melt.merge(df_cpi, on=['date', 'state'], how='left')
df_merged = df_merged.rename(columns={'state': 'city'})
df_merged = df_merged.rename(columns={'earnings': 'earnings_state'})

In [ ]:
# --- Load Excel (second tab: Data1) ---
df = pd.read_excel('home_prices.xlsx', sheet_name='Data1',
    skiprows=list(range(1, 10)))

df.columns = df.columns.str.lower()
df = df.rename(columns={df.columns[0]: 'date'})
df['date'] = pd.to_datetime(df['date'], errors='coerce')

In [ ]:
# Example list of main cities (all lowercase)
main_cities = ['sydney','melbourne','brisbane','adelaide','perth','hobart','darwin','canberra']

def clean_col(col):
    col_lower = col.lower().strip()
    
    # Keep date as is
    if 'date' in col_lower:
        return 'date'
    
    if 'median price' in col_lower:

        # Check for established houses
        if 'established house' in col_lower:
            for city in main_cities:
                if city in col_lower:
                    return f'house_{city}'
        
        # Check for attached dwellings
        if 'attached dwelling' in col_lower:
            for city in main_cities:
                if city in col_lower:
                    return f'attached_{city}'
    
    # If no match, just return the original (or None to drop)
    return None

# Compute new column names
new_cols = [clean_col(c) for c in df.columns]

# Identify which columns are valid (not None)
valid_cols = [i for i, c in enumerate(new_cols) if c is not None]

# Keep only those columns in the DataFrame
df = df.iloc[:, valid_cols]

# Assign the cleaned column names
df.columns = [new_cols[i] for i in valid_cols]

# --- Filter date range ---
df = df[df['date'] >= '2010-06-01']
display(df)

In [ ]:
# Identify columns
house_cols = [c for c in df.columns if c.startswith('house_')]
attached_cols = [c for c in df.columns if c.startswith('attached_')]
print(house_cols)
# Melt house prices
df_house = df.melt(id_vars='date', value_vars=house_cols,
                   var_name='city', value_name='house_price')
df_house['city'] = df_house['city'].str.replace('house_', '', regex=False)

# Melt attached prices
df_attached = df.melt(id_vars='date', value_vars=attached_cols,
                      var_name='city', value_name='attached_price')
df_attached['city'] = df_attached['city'].str.replace('attached_', '', regex=False)

# 6️⃣ Merge house + attached prices by date + city
df_long = pd.merge(df_house, df_attached, on=['date','city'], how='left')

In [ ]:
# Merge median house prices with earnings/CPI
df_full = pd.merge(
    df_long,         # the median house prices long table
    df_merged,       # your earnings/CPI table
    on=['date','city'],  # merge on date and city
    how='left'       # keep all rows from df_merged
)

display(df_full)

In [ ]:
#CPI adjustment
price_cols = ["house_price", "attached_price", "earnings_state"]

# For each city, find the row with the latest date and grab its CPI
ref_cpi = (
    df_full.sort_values("date")
      .groupby("city")
      .last()[["cpi"]]
      .rename(columns={"cpi": "ref_cpi"})
)
 
print("Reference CPI per city (latest date+city row):")
print(ref_cpi.to_string())
 
df_test = df_full.join(ref_cpi, on="city")
 
for col in price_cols:
    df_test[f"{col}_real"] = (df_test[col] * df_test["ref_cpi"] / df_test["cpi"]).round(2)
 
df_test = df_test.drop(columns=["ref_cpi"])

display(df_test)

In [ ]:
df_test['annual_earnings_real'] = df_test['earnings_state_real'] * 52
df_test['annual_earnings_nominal'] = df_test['earnings_state'] * 52
df_test['house_price_real'] = df_test['house_price_real'] * 1000
df_test['attached_price_real'] = df_test['attached_price_real'] * 1000
df_test['house_price'] = df_test['house_price'] * 1000
df_test['attached_price'] = df_test['attached_price'] * 1000

df_test['pti_house_real'] = df_test['house_price_real']/df_test['annual_earnings_real']
df_test['pti_attached_real'] = df_test['attached_price_real']/df_test['annual_earnings_real']

df_test['pti_house_nominal'] = df_test['house_price']/df_test['annual_earnings_nominal']
df_test['pti_attached_nominal'] = df_test['attached_price']/df_test['annual_earnings_nominal']

df_test[['city','date','house_price_real','annual_earnings_real','pti_house_real']].tail(10)

In [ ]:
df_test.to_csv("../docs/house_earnings_dataset.csv", index=False)